# z614 - Walk-forward 6 folds (Etapa 12)
Mismos features (FE609). Extiende z609 de 3 a 6 cortes, deslizando de a 1 mes (ventanas de 2 meses, se solapan). Mas lento que z609 (2x folds), pero validacion mas robusta.

In [1]:
!pip install -q lightgbm pyarrow optuna

In [2]:
import os
import numpy as np
import polars as pl
import lightgbm as lgb
import optuna
import warnings
warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

In [3]:
PARAM = {
    'experimento': 'LGB09_WF6',
    'kaggle_competition': 'labo-iii-2026-ba',
    'base_path': '/home/ds/exp/FE609/',
    'archivo_features': 'tb_features_FE609.parquet',
    'apredecir_path': '/home/ds/datasets/product_id_apredecir201912.txt',
    'horizonte_meses': 2,
    'periodo_ultimo_dato': 201912,
    'periodo_target_final': 202002,
    'semilla': 102103,
    'n_trials': 50
}

ruta = os.path.join('/home/ds/exp', PARAM['experimento'])
os.makedirs(ruta, exist_ok=True)
print(ruta)

/home/ds/exp/LGB09_WF6


## 1. Cargar features y armar target

In [4]:
def periodo_a_meses(periodo: int) -> int:
    return (periodo // 100) * 12 + (periodo % 100)

df = pl.read_parquet(os.path.join(PARAM['base_path'], PARAM['archivo_features']))
df = df.sort(["product_id", "periodo"])

H = PARAM['horizonte_meses']

df = df.with_columns(
    pl.col("tn").shift(-H).over("product_id").alias("tn_target")
)
df = df.with_columns(
    (pl.col("periodo_m") + H).alias("periodo_target_m")
)

df_valido = df.filter(pl.col("tn_target").is_not_null())

## 2. Definir los 6 cortes (deslizando de a 1 mes, ventanas de 2 meses)
El ultimo corte (train&le;201910, valid 201911-201912) coincide con el split original, para que el submit final sea comparable.

In [5]:
train_max_list = [201905, 201906, 201907, 201908, 201909, 201910]

folds = []
for train_max in train_max_list:
    valid_min = train_max + 1
    valid_max = train_max + 2

    m_train_max = periodo_a_meses(train_max)
    m_valid_min = periodo_a_meses(valid_min)
    m_valid_max = periodo_a_meses(valid_max)

    train_f = df_valido.filter(pl.col("periodo_target_m") <= m_train_max)
    valid_f = df_valido.filter(
        (pl.col("periodo_target_m") >= m_valid_min) & (pl.col("periodo_target_m") <= m_valid_max)
    )
    folds.append((train_f, valid_f))
    print(f"corte train<={train_max} valid={valid_min}-{valid_max}: train={train_f.height} valid={valid_f.height}")

corte train<=201905 valid=201906-201907: train=22758 valid=1778
corte train<=201906 valid=201907-201908: train=23645 valid=1788
corte train<=201907 valid=201908-201909: train=24536 valid=1806
corte train<=201908 valid=201909-201910: train=25433 valid=1816
corte train<=201909 valid=201910-201911: train=26342 valid=1811
corte train<=201910 valid=201911-201912: train=27249 valid=1827


## 3. Preparar matrices por fold

In [6]:
cols_excluir = {"tn", "tn_target", "tn_shift1", "periodo", "periodo_target_m", "nacimiento_m"}
features = [c for c in df.columns if c not in cols_excluir]
categoricas = [c for c in ["product_id", "cat1", "cat2", "cat3", "brand", "descripcion"] if c in features]

def a_pandas(tabla):
    pdf = tabla.select(features + ["tn_target"]).to_pandas()
    for c in categoricas:
        pdf[c] = pdf[c].astype("category")
    return pdf

datasets_por_fold = []
for train_f, valid_f in folds:
    train_pd = a_pandas(train_f)
    valid_pd = a_pandas(valid_f)

    X_train = train_pd[features]
    y_train = np.log1p(train_pd["tn_target"].clip(lower=0))
    X_valid = valid_pd[features]
    y_valid = np.log1p(valid_pd["tn_target"].clip(lower=0))

    dtrain = lgb.Dataset(X_train, label=y_train, categorical_feature=categoricas,
                          params={'feature_pre_filter': False})
    dvalid = lgb.Dataset(X_valid, label=y_valid, categorical_feature=categoricas, reference=dtrain,
                          params={'feature_pre_filter': False})
    datasets_por_fold.append((dtrain, dvalid))

## 4. Optuna sobre el promedio de los 6 folds

In [7]:
def objective(trial):
    params = {
        'objective': 'regression',
        'metric': 'rmse',
        'verbosity': -1,
        'seed': PARAM['semilla'],
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 15, 255),
        'min_data_in_leaf': trial.suggest_int('min_data_in_leaf', 5, 200),
        'feature_fraction': trial.suggest_float('feature_fraction', 0.5, 1.0),
        'bagging_fraction': trial.suggest_float('bagging_fraction', 0.5, 1.0),
        'bagging_freq': trial.suggest_int('bagging_freq', 1, 7),
        'lambda_l1': trial.suggest_float('lambda_l1', 1e-8, 10.0, log=True),
        'lambda_l2': trial.suggest_float('lambda_l2', 1e-8, 10.0, log=True),
        'max_depth': trial.suggest_int('max_depth', -1, 15),
    }

    scores = []
    for dtrain, dvalid in datasets_por_fold:
        modelo = lgb.train(
            params,
            dtrain,
            num_boost_round=2000,
            valid_sets=[dvalid],
            callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)]
        )
        scores.append(modelo.best_score['valid_0']['rmse'])

    return float(np.mean(scores))

study = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=PARAM['semilla']))
study.optimize(objective, n_trials=PARAM['n_trials'], show_progress_bar=True)

print("mejor rmse promedio (6 folds):", study.best_value)
print("mejores params:", study.best_params)

  0%|          | 0/50 [00:00<?, ?it/s]

mejor rmse promedio (6 folds): 0.4380260127784337
mejores params: {'learning_rate': 0.011122320048541079, 'num_leaves': 160, 'min_data_in_leaf': 28, 'feature_fraction': 0.6462356903686512, 'bagging_fraction': 0.6690271174922295, 'bagging_freq': 2, 'lambda_l1': 0.00027274983045861347, 'lambda_l2': 0.00019342480300206765, 'max_depth': -1}


## 5. Reentrenar final con el ultimo corte (train&le;201910, valid 201911-201912)

In [8]:
mejores_params = dict(study.best_params)
mejores_params.update({
    'objective': 'regression',
    'metric': 'rmse',
    'verbosity': -1,
    'seed': PARAM['semilla']
})

dtrain_final, dvalid_final = datasets_por_fold[-1]

modelo_final = lgb.train(
    mejores_params,
    dtrain_final,
    num_boost_round=2000,
    valid_sets=[dtrain_final, dvalid_final],
    valid_names=['train', 'valid'],
    callbacks=[lgb.early_stopping(stopping_rounds=100), lgb.log_evaluation(period=100)]
)

print("mejor iteracion:", modelo_final.best_iteration)

Training until validation scores don't improve for 100 rounds
[100]	train's rmse: 0.653993	valid's rmse: 0.702123
[200]	train's rmse: 0.417435	valid's rmse: 0.513193
[300]	train's rmse: 0.351503	valid's rmse: 0.483195
[400]	train's rmse: 0.318077	valid's rmse: 0.480558
Early stopping, best iteration is:
[388]	train's rmse: 0.321499	valid's rmse: 0.480539
mejor iteracion: 388


## 6. Prediccion para 202002 y submit

In [9]:
futuro = df.filter(pl.col("periodo") == PARAM['periodo_ultimo_dato'])
futuro_pd = futuro.select(features).to_pandas()
for c in categoricas:
    futuro_pd[c] = futuro_pd[c].astype("category")

pred_log = modelo_final.predict(futuro_pd, num_iteration=modelo_final.best_iteration)
pred_tn = np.expm1(pred_log)
pred_tn = np.clip(pred_tn, 0, None)

resultado = futuro.select(["product_id"]).to_pandas()
resultado["tn"] = pred_tn

In [10]:
apredecir = pl.read_csv(PARAM['apredecir_path'], separator="\t").to_pandas()

submit = apredecir[["product_id"]].merge(resultado, on="product_id", how="left")
print("nulos en submit (deberian ser 0):", submit["tn"].isna().sum())
submit["tn"] = submit["tn"].fillna(0.0)

archivo_submit = os.path.join(ruta, f"{PARAM['experimento']}_submit.csv")
submit.to_csv(archivo_submit, index=False)
print(archivo_submit)
submit.head()

nulos en submit (deberian ser 0): 0
/home/ds/exp/LGB09_WF6/LGB09_WF6_submit.csv


,product_id,tn
0,20001,1206.942659
1,20002,1095.137744
2,20003,796.228938
3,20004,594.561326
4,20005,555.915579


In [11]:
def kaggle_submit(competencia, archivo, mensaje):
    comando = f'kaggle competitions submit -c {competencia} -f {archivo} -m "{mensaje}"'
    os.system(comando)

kaggle_submit(PARAM['kaggle_competition'], archivo_submit, f"{PARAM['experimento']} walk-forward 6 folds")

100%|██████████| 18.7k/18.7k [00:00<00:00, 54.2kB/s]


91 submissions remaining today.
Successfully submitted to Labo III, 2026 BA

In [12]:
importancia = pl.DataFrame({
    "feature": modelo_final.feature_name(),
    "importancia": modelo_final.feature_importance(importance_type="gain")
}).sort("importancia", descending=True)

importancia.write_csv(os.path.join(ruta, "feature_importance.csv"))
print(os.path.join(ruta, "feature_importance.csv"))

/home/ds/exp/LGB09_WF6/feature_importance.csv
